In [ ]:
!pip install openai

In [2]:
from dotenv import load_dotenv

In [4]:
import os

load_dotenv()

api_key = os.environ["Open_router_api"]
print("API key loaded:", bool(api_key))

API key loaded: True


In [ ]:
import json
import time

from openai import OpenAI

# base_url MUST include /api/v1 — the SDK appends "/chat/completions" to it.
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"

# Must be a DIRECT link to an image file, on a host that allows programmatic
# fetching. upload.wikimedia.org rate-limits/blocks this network (403/429/400),
# so use a host that doesn't. Verified reachable: picsum.photos,
# raw.githubusercontent.com, images.unsplash.com.
IMAGE_URL = "https://picsum.photos/id/237/400/300"

print("Sending tiny test prompt to OpenRouter...")

for attempt in range(5):
    try:
        response = client.chat.completions.create(
          model=MODEL,
          max_tokens=3000,  # reasoning model: needs headroom for its thinking tokens
          messages=[
            {
              "role": "user",
              "content": [
                {
                  "type": "text",
                  "text": "What is in this image? Answer in 3 words max."
                },
                {
                  "type": "image_url",
                  "image_url": {
                    "url": IMAGE_URL
                  }
                }
              ]
            }
          ],
          extra_headers={
            "HTTP-Referer": "https://google.com",
            "X-Title": "Colab Multimodal Playground",
          }
        )

        # OpenRouter returns upstream provider failures as HTTP 200 with an
        # "error" key and choices=None. The SDK happily parses that, so
        # response.choices[0] raises "'NoneType' object is not subscriptable"
        # and hides the real reason. Always check before subscripting.
        if not response.choices:
            err = getattr(response, "error", None) or response.model_dump().get("error")
            print(f"\n⚠️  Attempt {attempt + 1}: no choices returned.")
            print(json.dumps(err, indent=2) if err else response.model_dump_json(indent=2)[:1000])
            if attempt < 4:
                time.sleep(5 * (attempt + 1))
                continue
            break

        print("\n✅ Success! Model Response:")
        print(response.choices[0].message.content)
        break

    except Exception as e:
        print(f"\n❌ Error during execution: {type(e).__name__}: {e}")
        break

In [ ]:
import base64
import json
import time

import requests

# Raw-HTTP variant with the image inlined as base64, so the provider never has
# to fetch a remote URL itself (removes one whole class of failure).

URL = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://google.com",
    "X-Title": "Colab Multimodal Playground"
}

img_resp = requests.get(IMAGE_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
img_resp.raise_for_status()
b64_image = base64.b64encode(img_resp.content).decode()
print(f"Image downloaded: {len(img_resp.content) / 1024:.1f} KB")

data = {
    "model": MODEL,
    "max_tokens": 3000,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What is in this image? Answer in 3 words max."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{b64_image}"
                    }
                }
            ]
        }
    ]
}

print("Sending direct HTTP POST request to OpenRouter...")

for attempt in range(5):
    response = requests.post(URL, headers=headers, data=json.dumps(data), timeout=300)
    print(f"\n--- Attempt {attempt + 1} | Status Code: {response.status_code} ---")

    try:
        response_json = response.json()
    except Exception as e:
        print(f"❌ Failed to parse JSON response: {e}")
        print("Raw output text:", response.text[:500])
        break

    # HTTP 200 does NOT mean success. The free NVIDIA endpoint frequently returns
    # 200 with an "error" body like:
    #   "Upstream error from Nvidia: ResourceExhausted: Worker local total
    #    request limit reached (16/16)"  code 502, provider_unavailable
    # That is transient capacity exhaustion on the free tier — just retry.
    if "error" in response_json:
        err = response_json["error"]
        print("⚠️  API error payload:")
        print(json.dumps(err, indent=2))
        if err.get("code") in (429, 502, 503) and attempt < 4:
            wait = 5 * (attempt + 1)
            print(f"Transient — retrying in {wait}s...")
            time.sleep(wait)
            continue
        break

    if "choices" in response_json:
        print("\n✅ Success! Model Response:")
        print(response_json["choices"][0]["message"]["content"])
        break

    print("❌ Unexpected payload:")
    print(json.dumps(response_json, indent=2)[:1000])
    break